## Imports

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time 

## Reading in Data

In [6]:
data = pd.read_csv('../Data/exon_ranges_summary.csv')

## Simulation function

Defaults creating a CSV to False. (so don't get a ton of .csv's)

work in progress! 

In [18]:

def get_results_dictionary(species_data, genome_size, num_active_te, range_1_start, range_1_end, range_2_start, range_2_end):
     # Dictionary to store the results
        species_results = {
            'Species': species_data['Species'],
            'Beginning_GENOME_SIZE': genome_size,
            'Active_tes': num_active_te,
            'range_1_start': range_1_start,
            'range_1_end': range_1_end,
            'range_2_start': range_2_start,
            'range_2_end': range_2_end,
            'TE_mobilized': 0,
            'TE_static': 0,
            'TE_in_exons': 0,
            'TE_in_non_coding': 0,
            'Exon_new_size': range_1_end - range_1_start + 1,
            'Non_coding_new_size': range_2_end - range_2_start + 1,
            'Total_Genome_growth': 0,
            'Total_time': 0
        }
        return species_results






def simulation(data, simulation_rounds=100, num_active_te=1000, te_mobilize_threshold=0.5, num_species=1, to_csv=False):
    mean_length = 5000 # avg. TE length
    std_dev_length = 2000 # std. dev. of TE length

    results_list = []
    
    for i in range(num_species):
        species_data = data.iloc[i]
        te_lengths = np.random.normal(loc=mean_length, scale=std_dev_length, size=num_active_te)
        te_lengths = np.clip(te_lengths, 100, 10000).astype(int)

        genome_size = species_data['Genome_size']
        average_range = species_data['Average Range']

        range_1_start, range_1_end = 0, average_range
        range_2_start, range_2_end = average_range + 1, genome_size

        results = get_results_dictionary(species_data, genome_size, num_active_te, range_1_start, range_1_end, range_2_start, range_2_end)

        start_time = time.time()

        for _ in range(simulation_rounds): 
            for _ in range(num_active_te):
                te_length = random.choice(te_lengths) # randomly select a TE length from normal distribution
                if random.random() < te_mobilize_threshold:
                    prob_exon = range_1_end / genome_size # probability of TE landing in exon
                    # te_position = random.randint(0, genome_size)
                    # print(f'TE position: {te_position}')

                    if random.random() < prob_exon: # lands in exon (range 1)
                        range_1_end += te_length # expand range 1
                        range_2_start = range_1_end + 1
                        range_2_end += te_length # expand range 2 
                        results['TE_in_exons'] += 1
                    else:
                        range_2_end += te_length # expand range 2
                        results['TE_in_non_coding'] += 1 # increment count for non-coding TEs
        
                    results['TE_mobilized'] += 1
                else:
                    results['TE_static'] += 1

        end_time = time.time()
        results['Exon_new_size'] = range_1_end - range_1_start + 1
        results['Non_coding_new_size'] = range_2_end - range_2_start + 1
        results['Total_Genome_growth'] = results['Exon_new_size'] + results['Non_coding_new_size'] - genome_size
        results['Total_time'] = end_time - start_time

        results_list.append(results)

    results_df = pd.DataFrame(results_list) # , index=[0])

    # save to DataFrame
    if to_csv:
        results_df.to_csv('CSV/simulation_results.csv', index=False)
    return results_df

## Run simulation for Sparrow Hawk + Giant Panda

can do other species or all species, just change what is being passed to function. Change csv to true below if you would like to see the .csv

In [19]:
sparrow_hawk_df = simulation(data, num_species=2, to_csv=False)
sparrow_hawk_df

TypeError: 'NoneType' object is not subscriptable

## View TE lengths